In [1]:
%load_ext autoreload
%autoreload 2

import os

os.chdir("../../")

In [2]:
import pandas as pd

consolidated_pairs = pd.read_csv('data/consolidated_pairs.csv')
print(f"Total rows loaded: {len(consolidated_pairs)}")

# Filter out rows with NaN pair_id (invalid rows)
consolidated_pairs = consolidated_pairs[consolidated_pairs['pair_id'].notna()].copy()
print(f"Valid pairs (after filtering): {len(consolidated_pairs)}")
print("\nFirst few rows:")
consolidated_pairs.head()

Total rows loaded: 34
Valid pairs (after filtering): 32

First few rows:


,pair_id,session_code,director_participant_code,director_time_started_utc,director_prolific_id,matcher_prolific_id,director_experiment_start_time,matcher_participant_code,matcher_time_started_utc,matcher_experiment_start_time,...,matcher_partner_understood,matcher_partner_adapted,matcher_collaboration_improved,matcher_partner_comment,matcher_partner_human_vs_ai,matcher_partner_human_vs_ai_why,matcher_ai_familiarity,matcher_ai_usage_frequency,matcher_ai_used_for_task,source_file
0,3kk7u0fy_08dnx2dx_ljtwijvs,3kk7u0fy,ljtwijvs,24:36.3,6562177a48308eebbb6359ea,5c28e9dfda51990001e3204b,2025-11-24T18:24:43.139397,08dnx2dx,18:42.9,2025-11-24T18:19:40.880931,...,5.0,5.0,5.0,They did great. They communicated the characte...,50.0,"One time they were saying things like ""lol"" wh...",moderately,rarely,no,all_apps_wide-2025-11-26 (1).csv
1,3kk7u0fy_2ifxj59u_qhyb02an,3kk7u0fy,2ifxj59u,38:01.1,67e8e0c0be5802df7311d109,67dac9fec33b7784afd3d08b,2025-11-24T17:38:14.518856,qhyb02an,04:22.0,2025-11-24T18:05:13.093778,...,4.0,5.0,5.0,my partner did excellent well beyond my expect...,98.0,"i think it was a human, judging from the vocab...",extremely,daily,yes,all_apps_wide-2025-11-26 (1).csv
2,3kk7u0fy_2ivt4mpj_g7urxzlh,3kk7u0fy,2ivt4mpj,30:14.5,5fd66ce8aec66457ff73d743,5ef1a79464e5837c95df3de8,2025-11-24T17:30:22.291596,g7urxzlh,28:15.3,2025-11-24T17:28:35.794256,...,5.0,5.0,5.0,I felt my partner was going faster than me at ...,25.0,My partner felt more AI due to how fast they w...,very,daily,yes,all_apps_wide-2025-11-26 (1).csv
3,3kk7u0fy_ddls1zzn_ldgryt73,3kk7u0fy,ldgryt73,30:12.2,666dbb1357a8cfb6025951df,6713eeea64ca1fb59cbcbce4,2025-11-24T17:30:22.306345,ddls1zzn,36:26.1,2025-11-24T17:36:33.223531,...,4.0,5.0,5.0,"I think they did well overall, in the 1st 2 ro...",100.0,The way they responded to me as well as the ve...,extremely,daily,no,all_apps_wide-2025-11-26 (1).csv
4,3kk7u0fy_ecjbn129_fkm1onm5,3kk7u0fy,fkm1onm5,03:12.4,67ace3fc5892a4d45463156b,66a276d13f40200c70c28109,2025-11-24T18:03:16.258846,ecjbn129,57:49.0,2025-11-24T17:57:55.990845,...,5.0,5.0,5.0,My partner adapted to the questions I was aski...,100.0,They described things the way a human would. I...,slightly,rarely,no,all_apps_wide-2025-11-26 (1).csv


In [3]:
import re

cleaned_transcripts = []

def get_clean_transcript(transcript: str) -> str:
    # Replace multiple whitespace with single space
    trainscript_refined = re.sub(r"\s+", " ", transcript)
    # Insert double newlines before timestamps
    trainscript_refined = re.sub(
        r"(\[(\d{2}:\d{2}:\d{2})\]\s)", r"\n\n\1", trainscript_refined
    ).strip()
    return trainscript_refined


for idx, row in consolidated_pairs.iterrows():
    pair_id = row['pair_id']
    
    # Skip if pair_id is NaN (shouldn't happen after filtering, but double-check)
    if pd.isna(pair_id):
        continue
    
    for round_num in range(1, 5):
        # Get transcript (use director or matcher - they're duplicates)
        director_col = f'round{round_num}_director_chat_transcript'
        matcher_col = f'round{round_num}_matcher_chat_transcript'
        
        transcript = row.get(director_col, '')
        if pd.isnull(transcript) or (isinstance(transcript, str) and len(transcript.strip().split()) <= 20):
            transcript = row.get(matcher_col, '')
        
        
        cleaned_transcripts.append(
            {
                "pair_id": pair_id,
                "round_ix": round_num,
                "transcript": get_clean_transcript(transcript)
            }
        )

cleaned_transcripts_df = pd.DataFrame(cleaned_transcripts)

In [4]:
from src.data.utils import pretty_print


pretty_print(cleaned_transcripts_df.sample(5))

In [5]:
cleaned_transcripts_df.to_csv("data/cleaned_human_to_human_transcripts.csv", index=False)

In [6]:
cleaned_transcripts_df.shape

(128, 3)